# Synthèse technique — Mise en place d'un Écosystème d'IA
**Cours** : Mise en place d'un écosystème d'IA — AEC (Spécialisation technique)
**Enseignant** : Dr. Komi Sodoké
**Date** : 06/06/2026

## Page de garde (remise finale)

**Programme** : AEC — Spécialisation technique en intelligence artificielle  
**Cours** : Mise en place d'un écosystème d'IA  
**Travail** : Projet synthèse — virtualisation et monitoring d'un écosystème IA  
**Enseignant** : Dr. Komi Sodoké  
**Étudiant** : Merzak , Miloud et Marouane Bouriel  
**Date de remise** : 06/06/2026

---

## Sommaire
1. [Résumé](#Résumé)
2. [Schémas techniques intégrés (version remise finale)](#Schémas-techniques-intégrés-(version-remise-finale))
3. [1. Dimensionnement et choix matériel](#1.-Dimensionnement-et-choix-matériel)
4. [2. Audit de l'environnement (Colab) — vérifications clés](#2.-Audit-de-l'environnement-(Colab)-—-vérifications-clés)
5. [3. Conteneurisation — Dockerfile recommandé](#3.-Conteneurisation-—-Dockerfile-recommandé)
6. [4. Monitoring & Watchdog — principes et script](#4.-Monitoring-&-Watchdog-—-principes-et-script)
7. [Annexes](#Annexes)


## Résumé
Ce notebook regroupe la synthèse technique du projet : dimensionnement matériel, audit d'environnement (Colab), conteneurisation (Dockerfile) et conception d'un agent de monitoring (watchdog). Les annexes contiennent le Dockerfile complet, le script Watchdog et la sortie de la simulation.

## Schémas techniques intégrés (version remise finale)

### Figure 1 — Architecture matérielle
<img src="images/architecture.png" alt="Architecture matérielle" width="950"/>
<div align="center"><em>Figure 1 — Vue d'ensemble CPU/GPU/PCIe/stockage/monitoring.</em></div>

### Figure 2 — Flux réseau
<img src="images/network_flow.png" alt="Flux réseau" width="950"/>
<div align="center"><em>Figure 2 — Flux de requêtes et de télémétrie entre services.</em></div>

### Figure 3 — Pipeline CI/CD
<img src="images/cicd_pipeline.png" alt="Pipeline CI/CD" width="950"/>
<div align="center"><em>Figure 3 — Chaîne Dev → CI → Registry → Staging → CD → Monitoring.</em></div>

> Ces schémas sont intégrés directement dans le notebook principal pour la remise finale.

## 1. Dimensionnement et choix matériel
Contexte : entraînement d'un grand modèle LLM.
- Option A : 1x NVIDIA A100 (40 Go), CPU 8 cœurs, 16 lignes PCIe Gen4.
- Option B : 4x NVIDIA RTX 4090 (24 Go chacun), CPU 16 cœurs, 24 lignes PCIe Gen4 (partagées).
**Recommandation** : Option A pour simplicité et grande VRAM unifiée; Option B si entraînement distribué et architecture NVLink/PCIe adaptée (CPU plus puissant recommandé).

## 2. Audit de l'environnement (Colab) — vérifications clés
Exécuter les commandes suivantes dans un notebook Colab pour valider l'environnement GPU :
```
!uname -a
!cat /proc/cpuinfo | grep "model name" | head -n 1
!nvidia-smi
```
Exemple de commande pour obtenir nom GPU, VRAM et température :

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,temperature.gpu --format=csv,noheader,nounits

## 3. Conteneurisation — Dockerfile recommandé
Le Dockerfile ci‑dessous est conçu pour exécuter des workloads PyTorch GPU basé sur CUDA runtime fourni par NVIDIA :
```dockerfile
FROM nvidia/cuda:12.1.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y \
    python3-pip \
    python3-dev \
    && rm -rf /var/lib/apt/lists/*
# Installer PyTorch et pandas (adapter la commande selon la version CUDA désirée)
RUN pip3 install torch pandas
WORKDIR /app
COPY train.py /app/
CMD ["python3", "train.py"]
```
Commentaires : pour garantir la compatibilité PyTorch/CUDA, préférez installer la roue officielle PyTorch correspondant à la version CUDA indiquée par `nvidia-smi` ou utilisez l'index recommandé par pytorch.org.

## 4. Monitoring & Watchdog — principes et script
Règles présentées :
- Température GPU > 82°C → alerte Thermal Throttling
- Erreurs ECC > 10 → basculement / isolation du nœud
Le script Python suivant implémente une simulation du watchdog (voir Annexe pour la sortie).

In [ ]:
import time
import random

def watchdog_logger(temps, temp_gpu, errors_ecc):
    print(f"[LOG - Sec {temps}] Température GPU: {temp_gpu}°C | Erreurs ECC: {errors_ecc}")
    if temp_gpu > 82:
        print("  --> [ALERT SYSTEM] Vitesse d'horloge bridée automatiquement (Thermal Throttling)!")
    if errors_ecc > 10:
        print("  --> [CRITICAL ERROR] Trop d'erreurs ECC ! Isolation du nœud et basculement du calcul.")
        return False
    return True

# Simulation (10 secondes)
for seconde in range(1, 11):
    temp_simulee = random.randint(70, 86)
    ecc_simulees = 0 if seconde < 6 else (seconde - 5) * 4
    if not watchdog_logger(seconde, temp_simulee, ecc_simulees):
        break
    time.sleep(1)

## Annexes
### Annexe A — Dockerfile complet
```dockerfile
FROM nvidia/cuda:12.1.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y \
    python3-pip \
    python3-dev \
    && rm -rf /var/lib/apt/lists/*
RUN pip3 install torch pandas
WORKDIR /app
COPY train.py /app/
CMD ["python3", "train.py"]
```

### Annexe B — Script Watchdog (fichier `scripts/watchdog_simulate.py`)
Le script complet est disponible dans `c:/Users/Public/Documents/TELECHARGEMENT/BdeB/scripts/watchdog_simulate.py`. Ci‑dessous le même contenu en cellule exécutable (voir cellule précédente).

### Annexe C — Exemple de sortie de la simulation (watchdog_output.txt)
```
[LOG - Sec 1] Température GPU: 71°C | Erreurs ECC: 0
[LOG - Sec 2] Température GPU: 81°C | Erreurs ECC: 0
[LOG - Sec 3] Température GPU: 79°C | Erreurs ECC: 0
[LOG - Sec 4] Température GPU: 85°C | Erreurs ECC: 0
  --> [ALERT SYSTEM] Vitesse d'horloge bridée automatiquement (Thermal Throttling)!
[LOG - Sec 5] Température GPU: 77°C | Erreurs ECC: 0
[LOG - Sec 6] Température GPU: 75°C | Erreurs ECC: 4
[LOG - Sec 7] Température GPU: 71°C | Erreurs ECC: 8
[LOG - Sec 8] Température GPU: 76°C | Erreurs ECC: 12
  --> [CRITICAL ERROR] Trop d'erreurs ECC ! Isolation du nœud et basculement du calcul.
```

### Annexe D — Document généré
Le document Word généré est disponible : `c:/Users/Public/Documents/TELECHARGEMENT/BdeB/Synthese_Ecosysteme_IA.docx`.